In [1]:
import os
import pandas  as pd


In [3]:
import pysam
import pandas as pd
chr_list = ['chr1', 'chr2', 'chr3', 'chr4','chr5',
            'chr6', 'chr7', 'chr8', 'chr9', 'chr10',
            'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
            'chr16', 'chr17', 'chr18', 'chr19', 'chr20',
            'chr21', 'chr22'
            ]
def read_vcf(vcf_path):
    vcf_reader = pysam.VariantFile(vcf_path)
    records = []
    # try:
    for record in vcf_reader:
        # print(record)
        if record.chrom in chr_list:
            record_dict = {
                "CHROM": record.chrom,
                "POS": record.pos,
                "ID": record.id,
                "REF": record.ref,
                "QUAL": record.qual,
                "FILTER": record.filter,
                "INFO": record.info,
                "TYPE": record.info['SVTYPE']
            }
            
            records.append(record_dict)
    # except:
    #     i=0

    return pd.DataFrame(records)

vcf2 = '/mnt/nas/Bio/VinhDC/vn920_cnnlsv/vn920.vcf'
test = read_vcf(vcf2)
len(test)
pysam.VariantFile(vcf2)

In [11]:
test['TYPE']

NameError: name 'test' is not defined

In [6]:
import os
import subprocess
'''
Calculate True Positives, False Positives, and False Negatives:
True positives: SVs present in both VCF files and overlapping according to the defined criteria.
False positives: SVs present in one VCF file but not the other, or present in both but not overlapping.
False negatives: SVs present in one VCF file but not the other.
'''

survivor_path = '/mnt/WD/VGP/VN_007_920_benchmark/SURVIVOR/Debug/SURVIVOR'
# 1000 2 1 1 0 50 cnnlsv_survivor.vcf

# /mnt/WD/VGP/VN_007_920_benchmark/SURVIVOR/Debug/SURVIVOR merge /mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/output/input_survivor.txt 1000 2 1 1 0 50 cnnlsv_survivor.vcf
def calculate_precision_recall(vcf1, vcf2):
    """Calculates precision and recall based on an overlap file generated by SURVIVOR.

    Args:
    vcf1: Path to the first VCF file.
    vcf2: Path to the second VCF file.

    Returns:
    tuple: Precision and recall values.
    """
    fileNameVCF1 = os.path.basename(vcf1)
    fileNameVCF2 = os.path.basename(vcf2)
    with open("list_file_VCF.txt", 'w') as fp:
        fp.write("\n".join([vcf1, vcf2, ""]))
    output_file = "_".join([fileNameVCF1, fileNameVCF2])
    subprocess.run([survivor_path, "merge", "list_file_VCF.txt", "1000", "1 ", "1", "1", "0", "50", output_file])
    print("Done")
    df = read_vcf(output_file)
    # Parse overlap file to count true positives, false positives, and false negatives
    frequency = df['SUPP_VEC'].value_counts()
    true_positives = df[df['QUAL'] >= 100]['SUPP_VEC'].value_counts()['11']
    false_positives =  df[df['QUAL'] < 100]['SUPP_VEC'].value_counts()['11']
    false_negatives = frequency['01'] + frequency['10']


    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    return precision, recall, df

    return df
    

    # Calculate precision and recall

# Example usage
# overlap_file = "overlaps.txt"
# subprocess.run(["survivor", "-i", vcf1, vcf2, "-o", overlap_file])
# precision, recall = calculate_precision_recall(vcf1, vcf2, overlap_file)


In [7]:
vcf1 = '/mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/output/cnnlsv_survivor.sorted.vcf'
vcf2 = '/mnt/WD/VGP/VN_007_920_benchmark/SURVIVOR/Debug/vn007.vcf'

precision, recall, df = calculate_precision_recall(vcf1, vcf2)
print("Precision:", precision)
print("Recall:", recall)


VCF Parser: could not open file: /mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/output/cnnlsv_survivor.sorted.vcf
[W::vcf_parse_info] INFO/END=1505 is smaller than POS at chr1:883245


Done
Precision: 0.800957228595305
Recall: 0.6183577712609971


In [47]:
vcf1 = '/mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/output/cutesv.VN007.merge.hifi_reads_map-hifi.minimap2.sorted.cnnLSV.test.vcf'
vcf2 = '/mnt/WD/VGP/VN_007_920_benchmark/SURVIVOR/Debug/vn007.vcf'

precision, recall, df = calculate_precision_recall(vcf1, vcf2)
print("Precision:", precision)
print("Recall:", recall)


cutesv.VN007.merge.hifi_reads_map-hifi.minimap2.sorted.cnnLSV.test.vcf_vn007.vcf
['/mnt/WD/VGP/VN_007_920_benchmark/SURVIVOR/Debug/SURVIVOR', 'merge', 'list_file_VCF.txt', '1000 1 1 1 0 50', 'cutesv.VN007.merge.hifi_reads_map-hifi.minimap2.sorted.cnnLSV.test.vcf_vn007.vcf']
merging entries: 10870
merging entries: 41650
Done
Precision: 0.9512893982808023
Recall: 0.24267782426778242


In [28]:
frequency = df['SUPP_VEC'].value_counts()
frequency['11']

27970

In [42]:
(df.QUAL > 100).sum()

14501

In [13]:
vcf1 = '/mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/temp/cnnLSV/output/cnnlsv_survivor.sorted.vcf'
df = read_vcf(vcf1)


In [14]:
df['TYPE'].value_counts()

TYPE
INS    13689
DEL    10784
DUP     3232
TRA     1712
INV      113
Name: count, dtype: int64

In [8]:
vcf1 = '/mnt/nas/Bio/VinhDC/vn920_cnnlsv/cutesv.VN920.merge.hifi_reads_map-hifi.minimap2.sorted.vcf.reheader.setId.filterd.cnnlsv.vcf'
vcf2 = '/mnt/nas/Bio/VinhDC/vn920_temp/cutesv.VN920.merge.hifi_reads_map-hifi.minimap2.sorted.vcf.reheader.setId.filterd.vcf'

df = read_vcf(vcf2)
print(df['TYPE'].value_counts())

df = read_vcf(vcf1)
print(df['TYPE'].value_counts())

TYPE
INS    13821
DEL    11598
BND      389
DUP       87
INV       82
Name: count, dtype: int64


In [19]:
df['INFO'][0]['SVTYPE']

'DUP'